In [ ]:
## Import Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## Load Data
personality = pd.read_csv("vaping_sleep_data.csv")
new_personality = pd.read_csv("new_intake.csv")

stress = pd.read_csv("sleep_duration.csv")
new_stress = pd.read_csv("new_sleep_duration.csv")

hrv = pd.read_csv("HRV_data.csv")
sleep_summary = pd.read_csv("sleep_summary.csv")

## Data Cleaning
personality["pID"] = personality["pID"].replace({"O25": "025"})
new_personality["pID"] = new_personality["pID"].replace({
    "24037478": "111",
    "uceey12@ucl.ac.uk": "261"})

personality["pID"] = pd.to_numeric(personality["pID"], errors="coerce")
new_personality["pID"] = pd.to_numeric(new_personality["pID"], errors="coerce")
personality_all = pd.concat([personality, new_personality])

# Merging multiple datasets
stress_evening = stress[stress["ToD"] == "Evening"]
new_stress_evening = new_stress[new_stress["ToD"] == "Evening"]
stress_all = pd.concat([stress_evening, new_stress_evening])

# Create weekly stress averaging
stress_avg = (
    stress_all
    .groupby(["pID","study_week"])
    .agg(
        average_overall_stress=("stress_today_pm","mean"),
        average_compared_stress=("stress_vs_usual_pm","mean")
    )
    .reset_index())
# Personality Grouping using quantiles
personality_all["N_group"] = pd.qcut(
    personality_all["BFI10_N"],
    q=2,
    labels=["Low","High"])    

personality_all["C_group"] = pd.qcut(personality_all["BFI10_C"],2,labels=["Low","High"])
personality_all["O_group"] = pd.qcut(personality_all["BFI10_O"],2,labels=["Low","High"])

# More Merging
analysis_df = pd.merge(
    stress_avg,
    personality_all,
    on="pID",
    how="inner")

# Visualisation 
sns.boxplot(
    data=analysis_df,
    x="N_group",
    y="average_overall_stress",
    hue="study_week")

plt.title("Stress Levels by Neuroticism Group")
plt.xlabel("Neuroticism Group")
plt.ylabel("Average Stress")
plt.show()  